In [13]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

"Pierre-Auguste Renoir"
"Käthe Kollwitz"
"Max Ernst"
"Karel Appel"

In [16]:
Artist_name ="Karel Appel"

In [17]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_{Artist_name.split(" ")[-1]}.xlsx")
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_{Artist_name.split(" ")[-1]}.xlsx")
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_{Artist_name.split(" ")[-1]}.xlsx")

In [18]:
gemini_label.shape

(1000, 10)

In [19]:
full_df = claude_label.merge(gemini_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_gemini"))

In [20]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_gemini,artistic_value_comment_gemini,creativity_answer_gemini,creativity_comment_gemini
0,89,Utan titel,Karel,Appel,1956,Dutch,High,Appel's work is characterized by naïve formal ...,Yes,The CoBrA movement advocated spontaneous expre...,High,"Karel Appel's ""Utan titel"" (Untitled) from 195...",Yes,"""Utan titel"" (Untitled) by Karel Appel from 19..."
1,311,Flowers as a still life,Karel,Appel,1978,Dutch,authoritative commentary on this Karel Appel w...,Recommendation:** I suggest consulting:\n1. Th...,for authoritative commentary on this Karel App...,Recommendation:** I suggest consulting:\n1. Th...,High,"Karel Appel's ""Flowers as a Still Life,"" creat...",Yes,"Karel Appel's ""Flowers as a Still Life"" demons..."
2,800,TIGERBIRD,Karel,Appel,1952,Dutch,I encountered a technical issue with the web s...,Museum documentation** from institutions holdi...,but I encountered a technical issue with the w...,Museum documentation** from institutions holdi...,High,"Karel Appel's ""Tigerbird"" (1952) is widely reg...",Yes,"""Tigerbird"" is a testament to Karel Appel's cr..."
3,863,Screaming animal,Karel,Appel,1954,Dutch,"authoritative commentary on Karel Appel's ""Scr...","To complete your request properly**, I would r...",for authoritative commentary on Karel Appel's ...,"To complete your request properly**, I would r...",High,"Karel Appel's ""Screaming animal"" from 1954 is ...",Yes,"""Screaming animal"" exemplifies Karel Appel's c..."
4,1154,Untitled Composition,Karel,Appel,1983,Dutch,High,"Karel Appel, co-founder of the renowned CoBrA ...",Yes,"Appel's artistic philosophy embodied ""continuo...",High,"Karel Appel's ""Untitled Composition"" from 1983...",Yes,"Karel Appel's ""Untitled Composition"" from 1983..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,427171969,Sans titre,Karel,Appel,1971,Dutch,High,Karel Appel was an influential Dutch painter w...,Yes,"Like Jean Dubuffet, Appel found inspiration in...",High,"Karel Appel's ""Sans titre"" from 1971 is recogn...",Yes,"Karel Appel's ""Sans titre"" (1971) demonstrates..."
996,427174725,Earthbird,Karel,Appel,1955,Dutch,High,Karel Appel was one of the founders of the ava...,Yes,"Appel demonstrated commitment to expressive, s...",High,"Karel Appel's ""Earthbird"" (1955) is widely reg...",Yes,"""Earthbird"" by Karel Appel demonstrates signif..."
997,427174782,Looking through the window (Kykend door het ve...,Karel,Appel,1981,Dutch,authoritative commentary on this specific artw...,Limitation:** I cannot locate sufficient peer-...,for authoritative commentary on this specific ...,Limitation:** I cannot locate sufficient peer-...,High,"Karel Appel's ""Looking through the window (Kyk...",Yes,"Karel Appel's ""Looking through the window (Kyk..."
998,427186249,Untitled,Karel,Appel,1974,Dutch,High,Karel Appel was an influential Dutch painter w...,Yes,Appel's 1974 practice demonstrates genuine cre...,High,"Karel Appel's 1974 artwork, ""Untitled,"" is wid...",Yes,"Karel Appel's ""Untitled"" (1974) demonstrates s..."


In [21]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [22]:
claude_embed = np.load(f"clip_embeddings_claude_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)
gemini_embed = np.load(f"clip_embeddings_gemini_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)
openai_embed = np.load(f"clip_embeddings_openai_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)

# Golden Set

Golden set is a set of samples that have the same answers for "type" and "creative". To add to confidence, only samples with cosine similarity above a given threshold are kept.

In [37]:
threshold = 0.65

In [38]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [39]:
np.sum(consistent_artist)

np.int64(643)

In [40]:
np.sum(consistent_creative)

np.int64(598)

In [41]:
np.sum(consistent_overall)

np.int64(594)

In [42]:
checking=full_df.copy()
checking["creative_consist"]=consistent_creative
checking["artistic_consist"]=consistent_artist
checking["overall_consist"]=consistent_overall

In [43]:
print(f"""
For type, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
""")

print(f"""
For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
""")


For type, there are 643 samples that are consistent.


For creative, there are 598 samples that are consistent.



In [44]:
print(f"""
When looking at only type and creative, there are {np.sum(checking["overall_consist"])} samples that are consistent.
""")


When looking at only type and creative, there are 594 samples that are consistent.



In [45]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > threshold).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > threshold).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: End")

2025-12-16 05:13:22: Start
2025-12-16 05:13:22: Currently at 0
2025-12-16 05:13:22: End


In [46]:
checking["embed_artistic_consist"]=embed_consistent_artistic
checking["embed_creative_consist"]=embed_consistent_creative
checking["embed_overall_consist"]=embed_consistent_overall

In [51]:
print(f"""
[Easy] For artistic, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
[Hard] For artistic, there are {checking[(checking["artistic_consist"]==1) & (checking["embed_artistic_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy] For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
[Hard] For creative, there are {checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy]For overall, there are {np.sum(checking["overall_consist"])} samples that are consistent.
[Hard] For overall, there are {checking[(checking["overall_consist"]==1) & (checking["embed_overall_consist"]==1)].shape[0]} samples that are consistent.
""")


[Easy] For artistic, there are 643 samples that are consistent.
[Hard] For artistic, there are 483 samples that are consistent.


[Easy] For creative, there are 598 samples that are consistent.
[Hard] For creative, there are 16 samples that are consistent.


[Easy]For overall, there are 594 samples that are consistent.
[Hard] For overall, there are 12 samples that are consistent.



In [52]:
golden_set_move_creative = checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)]

In [53]:
golden_set_move_creative.shape

(16, 24)

In [54]:
golden_set_move_creative.to_excel(f"golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx",index=False)

# Double Check

In [22]:
golden_set_move_creative = pd.read_excel("golden_set_move_creative_65.xlsx")

In [23]:
golden_set_move_creative

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,...,artistic_value_answer,artistic_value_comment,creativity_answer,creativity_comment,creative_consist,artistic_consist,overall_consist,embed_artistic_consist,embed_creative_consist,embed_overall_consist
0,1596,Le peintre et son modèle,Pablo,Picasso,1964,Spanish,High,"In Picasso's later years, the theme of painter...",Yes,The work demonstrates residual Cubism through ...,...,High,"""Le Peintre et Son Modèle"" is a significant wo...",Yes,"In ""Le Peintre et Son Modèle,"" Picasso innovat...",1,1,1,1,1,1
1,1781,Verre et citron,Pablo,Picasso,1944,Spanish,High,"According to art historian John Richardson, ""t...",Yes,The work belongs to a series of small-scale wo...,...,High,"""Verre et citron"" is a notable example of Pica...",Yes,"""Verre et citron"" exemplifies Picasso's innova...",1,1,1,1,1,1
2,2849,HOMME ASSIS,Pablo,Picasso,1969,Spanish,High,"""Homme Assis"" was painted during Picasso's mos...",Yes,Picasso's objective to paint 'nature' contrast...,...,High,"Pablo Picasso's ""Homme Assis"" (1969) is a sign...",Yes,"""Homme Assis"" exemplifies Picasso's innovative...",1,1,1,1,1,1
3,3100,"Femme assise dans un fauteuil tressé, en gris ...",Pablo,Picasso,1953,Spanish,High,This portrait of Françoise Gilot was painted i...,Yes,The 1953 portrait innovates beyond Picasso's e...,...,High,"""Femme assise dans un fauteuil tressé, en gris...",Yes,"Picasso's ""Femme assise dans un fauteuil tress...",1,1,1,0,1,0
4,3483,DEUX HIRONDELLES,Pablo,Picasso,1932,Spanish,High,"Painted on May 14, 1932 at the height of his c...",Yes,The work demonstrates creativity through surpr...,...,High,"Pablo Picasso's 1932 painting ""Deux Hirondelle...",Yes,"""Deux Hirondelles"" exemplifies Picasso's creat...",1,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1741,424922090,Visage de femme,Pablo,Picasso,1953,Spanish,High,"This ceramic work, likely depicting Jacqueline...",Yes,"Picasso was inspired by those around him, with...",...,High,"""Visage de femme"" (1953) is a glazed ceramic p...",Yes,"""Visage de femme"" demonstrates Picasso's creat...",1,1,1,1,1,1
1742,424922091,Visage d'homme,Pablo,Picasso,1953,Spanish,High,While specific scholarly critique of this 1953...,Yes,Picasso's use of the cast shadow as a pictoria...,...,High,"""Visage d'homme"" (1953) exemplifies Picasso's ...",Yes,"""Visage d'homme"" showcases Picasso's continuou...",1,1,1,1,1,1
1743,424924633,Vase aztèque aux quatre visages,Pablo,Picasso,1957,Spanish,High,Vase Aztèque aux quatre visages captures the s...,Yes,This work captures Picasso's restless need to ...,...,High,"Pablo Picasso's ""Vase Aztèque aux Quatre Visag...",Yes,"Picasso's ""Vase Aztèque aux Quatre Visages"" de...",1,1,1,1,1,1
1744,424927848,Mousquetaire,Pablo,Picasso,1969,Spanish,High,"Between 1966 and 1972, Picasso displayed porte...",Yes,"For Picasso, the musketeer signified the golde...",...,High,"Pablo Picasso's 1969 painting ""Mousquetaire"" e...",Yes,"In ""Mousquetaire,"" Picasso showcases significa...",1,1,1,1,1,1
